# 1. Environment Setup & Data Loading

In [ ]:
import numpy as np 
import pandas as pd
import os
import kagglehub
path = kagglehub.competition_download('playground-series-s6e3')
print("Path to competition files:", path)
print("--------------------------------------------------------------------------------")
for dirname, _, filenames in os.walk('/kaggle/input'): # read and listing out all the file under kaggle/input path
    for filename in filenames:
        print(os.path.join(dirname, filename))
print("--------------------------------------------------------------------------------")
print("Load and assigning dataset.....")

def load_csv(path, name="dataset"): # Functionfor try and catch error for loading csv in the notebook
    try:
        df = pd.read_csv(path)
        print(f"Success - {name} loaded successfully: {df.shape}")
        return df

    except FileNotFoundError:
        print(f"Failed - {name} not found: {path}")
        return None

    except pd.errors.EmptyDataError:
        print(f"Failed - {name} is empty: {path}")
        return None

    except pd.errors.ParserError:
        print(f"Failed - Could not parse {name}: {path}")
        return None

    except Exception as e:
        print(f"Failed - Error loading {name}: {e}")
        return None

train_raw = load_csv( # loading training dataset
    "/kaggle/input/competitions/playground-series-s6e3/train.csv",
    "train_raw"
)

test_raw = load_csv( # loading testing dataset
    "/kaggle/input/competitions/playground-series-s6e3/test.csv",
    "test_raw"
)

# 2. Explotanary Data ( EDA )

## 2.1 Examine the dataset

In [ ]:
print(f"============================= Train Set =============================")
train_raw.info()
print(f"=====================================================================")
train_raw.head()

## 2.2 EDA - test_set

In [ ]:
test_raw.info()

In [ ]:
test_raw.head()

## 2.4 Identifying target columns 

In [ ]:
print("Train columns:")
print(train_raw.columns.tolist())

print("\nTest columns:")
print(test_raw.columns.tolist())

target_candidates = list(set(train_raw.columns) - set(test_raw.columns))

print("\nPossible target column(s):", target_candidates)

## 2.5 Distribution of numeric columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Select numeric columns and exclude ID
numeric_cols = train_raw.select_dtypes(include="number").columns.drop(
    ["id", "SeniorCitizen"], errors="ignore"
)

# Plot settings
n_cols = 3
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4 * n_rows)
)

# Make axes iterable even if there is only one subplot
axes = axes.flatten()

# Plot histogram + KDE for each numeric feature
for i, col in enumerate(numeric_cols):
    data = train_raw[col].dropna()

    sns.histplot(
        data,
        bins=30,
        kde=True,
        ax=axes[i],
        stat="density",
        edgecolor="black"
    )

    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Density")

# Hide unused subplots
for i in range(len(numeric_cols), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## 2.6 Distribution of categorical columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_raw["SeniorCitizen"] = train_raw["SeniorCitizen"].astype("category") # assign seniorcitizen column as category type instead of numeric
test_raw["SeniorCitizen"] = test_raw["SeniorCitizen"].astype("category")

categorical_cols = train_raw.select_dtypes(
    include=["object", "category", "bool"]
).columns.drop("id", errors="ignore")\



plt.figure(figsize=(20, 15))

for i, col in enumerate(categorical_cols):
    plt.subplot(5, 4, i + 1)

    top_categories = train_raw[col].value_counts().head(10).index

    plot_data = train_raw[
        train_raw[col].isin(top_categories)
    ]

    sns.countplot(
        x=plot_data[col],
        hue=plot_data[col],
        order=top_categories,
        palette="viridis",
        legend=False
    )

    plt.title(f"Count of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 3. Data processing

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make a copy so we don't modify train_raw
corr_data = train_raw.copy()

# Drop ID
corr_data = corr_data.drop(columns=["id"], errors="ignore")

# Convert categorical columns to numeric
categorical_cols = corr_data.select_dtypes(
    include=["object", "category", "bool"]
).columns

for col in categorical_cols:
    corr_data[col] = corr_data[col].astype("category").cat.codes

# Calculate correlation matrix
corr_matrix = corr_data.corr()

# Plot
plt.figure(figsize=(16, 12))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Matrix - Numeric & Categorical Features")
plt.tight_layout()
plt.show()

# 4. Baseline model

In [ ]:
# ============================================================
# 1. TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Separate features and target
X = train_raw.drop(columns=["Churn", "id"], errors="ignore")
y = train_raw["Churn"]

# Split BEFORE preprocessing/model training
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)
print(f"Full dataset     : {X.shape}")
print(f"Training set     : {X_train.shape}")
print(f"Validation set  : {X_valid.shape}")
print(f"Training target : {y_train.shape}")
print(f"Validation target: {y_valid.shape}")
print("=" * 70)


# ============================================================
# 2. BASIC PREPROCESSING
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify columns from TRAINING SET only
numeric_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# ============================================================
# 3. BASELINE MODEL
# ============================================================

from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# ============================================================
# 4. TRAIN BASELINE MODEL
# ============================================================

print("\nTRAINING BASELINE MODEL........")
baseline_model.fit(X_train, y_train)

# ============================================================
# 5. VALIDATION PREDICTIONS
# ============================================================

y_pred = baseline_model.predict(X_valid)
y_prob = baseline_model.predict_proba(X_valid)[:, 1]

# ============================================================
# 6. BASELINE EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("\n" + "=" * 70)
print("BASELINE RESULTS")
print("=" * 70)

print(f"Accuracy : {accuracy_score(y_valid, y_pred):.4f}")
print(f"Precision: {precision_score(y_valid, y_pred, pos_label='Yes'):.4f}")
print(f"Recall   : {recall_score(y_valid, y_pred, pos_label='Yes'):.4f}")
print(f"F1 Score : {f1_score(y_valid, y_pred, pos_label='Yes'):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_valid, y_prob):.4f}")

print("=" * 70)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_valid,
    y_pred,
    display_labels=["No", "Yes"]
)

plt.title("Baseline Confusion Matrix")
plt.show()